# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step workflow for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
This dataset is described by a Croissant schema and can be accessed via a public URL.

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print("\nDescription:")
print(metadata.description)
print("\nPublished:", getattr(metadata, 'datePublished', 'N/A'))
print("Identifier:", getattr(metadata, 'identifier', 'N/A'))

# Print dataset-level fields such as available record sets
record_sets = metadata.recordSet if hasattr(metadata, 'recordSet') else []
if record_sets:
    print(f"\nRecord set count: {len(record_sets)}")
else:
    print("\nNo explicit record sets listed in metadata; attempting to automatically detect record sets.")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

A record set collects related records in the dataset. We'll list record sets and fields with their `@id`s so we can reference them programmatically.

In [ ]:
# Helper to print recordsets, fields, and columns by @id
print("\nRecord sets overview:")
rs_overview = []
if record_sets:
    for rs in record_sets:
        rs_id = getattr(rs, '@id', None)
        rs_name = getattr(rs, 'name', None)
        print(f"- RecordSet: {rs_name} (@id={rs_id})")
        fields = getattr(rs, 'field', [])
        if not isinstance(fields, list):
            fields = [fields]
        if fields:
            print("  Fields:")
            for f in fields:
                print(f"    - {getattr(f, 'name', '?')} (@id={getattr(f, '@id', '?')})")
        columns = getattr(rs, 'column', [])
        if not isinstance(columns, list):
            columns = [columns]
        if columns:
            print("  Columns:")
            for c in columns:
                print(f"    - {getattr(c, 'name', '?')} (@id={getattr(c, '@id', '?')})")
        rs_overview.append(rs_id)
else:
    # Many FAIR2 datasets use a single, default record set detected dynamically
    # List available record set IDs using dataset.records() generator
    print("No record sets defined. Attempting to detect available record sets via API...")
    try:
        # Try iterating record sets by ID
        possible_rs_ids = dataset.record_set_ids
        print("Record sets found:")
        for rs_id in possible_rs_ids:
            print("- RecordSet @id:", rs_id)
        rs_overview = list(possible_rs_ids)
    except Exception as e:
        print("Could not programmatically determine record sets:", e)
        rs_overview = []

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All entities must be referenced by their `@id` (not name or order!).

In [ ]:
# Automatically discover record set IDs using the public Croissant schema
if not rs_overview:
    # Fallback: search for all available record set ids
    try:
        rs_overview = dataset.record_set_ids
    except Exception as e:
        rs_overview = []

# We'll now load all detected record sets to Pandas DataFrames
dataframes = {}
for rs_id in rs_overview:
    print(f"\nLoading records from RecordSet: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    print(f"  --> Loaded records: {len(df)} rows, {len(df.columns)} columns")
    print("  Columns:", df.columns.tolist())
    dataframes[rs_id] = df

# For demonstration, pick the first record set as our example below
if dataframes:
    main_rs_id = next(iter(dataframes.keys()))
    print(f"\nExamining RecordSet: {main_rs_id}")
    print(dataframes[main_rs_id].head())
else:
    main_rs_id = None
    print("No record sets could be loaded to DataFrames.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, or grouping by an attribute.

**Remember:** All field selection must use their exact column or field `@id` where possible, as inferred above.

In [ ]:
# Choose a numeric field (by column @id). For demonstration, we'll guess common protein-related fields.
# First, list columns to pick a numeric column.
if main_rs_id and main_rs_id in dataframes:
    df = dataframes[main_rs_id].copy()
    print(f"Columns in RecordSet {main_rs_id}:")
    for col in df.columns:
        print(f"  - {col}")

    # Choose a numeric field by inspecting columns (e.g., coverage, peptide count, MW, abundance, etc.)
    # Suppose '@id' for normalized abundance is 'normalized_abundance', can be replaced if naming differs
    import numpy as np
    numeric_candidates = [col for col in df.columns if df[col].dtype in (np.float64, np.int64) or np.issubdtype(df[col].dtype, np.number)]
    if not numeric_candidates:
        # attempt float-conversion for all columns
        for col in df.columns:
            try:
                df[col+'_float_test'] = pd.to_numeric(df[col], errors='coerce')
                num_null = df[col+'_float_test'].isnull().sum()
                if num_null < len(df):
                    numeric_candidates.append(col)
            except:
                continue
        # Remove temporary columns
        df = df[[c for c in df.columns if not c.endswith('_float_test')]]

    print("\nNumeric columns detected:", numeric_candidates)
    # Choose first available numeric field
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
    else:
        numeric_field = None

    # Filtering and normalization demo
    if numeric_field:
        # Convert column to float (if needed)
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 10
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"\nFiltered records with {numeric_field} > {threshold:.3f}:")
        print(filtered_df[[numeric_field]].head())

        # Normalize
        filtered_df[numeric_field + "_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, numeric_field + "_normalized"]].head())

        # Try grouping by a categorical field (e.g., description, accession, etc.)
        non_numeric_candidates = [col for col in df.columns if col != numeric_field]
        if non_numeric_candidates:
            group_field = non_numeric_candidates[0]
            print(f"\nGrouping by {group_field} and showing mean of {numeric_field}:\n")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head())
        else:
            print("Could not identify a categorical field for grouping.")
    else:
        print("No numeric fields available for analysis.")
else:
    print("No main record set loaded. Unable to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the selected numeric field and show a grouped plot by a categorical variable.

In [ ]:
import matplotlib.pyplot as plt

if main_rs_id and main_rs_id in dataframes and numeric_field:
    df = dataframes[main_rs_id]
    plt.figure(figsize=(8,4))
    # Numeric distribution
    plt.hist(df[numeric_field].dropna(), bins=50, alpha=0.7)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If we also have a group field, visualize means
    if 'group_field' in locals() and group_field and group_field in df.columns:
        group_means = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).head(10)
        plt.figure(figsize=(10,4))
        group_means.plot(kind='bar')
        plt.title(f"Top 10 {group_field} by mean {numeric_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()

## 6. Conclusion
In this notebook, we've:
- Loaded the FAIR² dataset from a Croissant schema using `mlcroissant`.
- Explored metadata for record sets and fields, referencing all entities by their canonical `@id`.
- Loaded all available record sets and demonstrated EDA by filtering and normalizing a numeric field.
- Visualized data distributions and explored possible groupings in the records.

You can now extend this workflow to analyze other record sets, fields, or perform deeper domain analysis.